In [1]:
import pandas as pd 
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

from pathlib import Path

## LOADING DATA

In [2]:
path_to_repo = Path('..').resolve()
path_to_data = path_to_repo / 'data'

In [3]:
application_record = pd.read_csv(path_to_data / 'application_record.csv')
credit_record = pd.read_csv(path_to_data / 'credit_record.csv')

In [4]:
application_record.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


# PREPROCESSING

In [5]:
application_record = application_record.drop_duplicates("ID")

In [6]:
application_record.nunique()

ID                     438510
CODE_GENDER                 2
FLAG_OWN_CAR                2
FLAG_OWN_REALTY             2
CNT_CHILDREN               12
AMT_INCOME_TOTAL          866
NAME_INCOME_TYPE            5
NAME_EDUCATION_TYPE         5
NAME_FAMILY_STATUS          5
NAME_HOUSING_TYPE           6
DAYS_BIRTH              16379
DAYS_EMPLOYED            9406
FLAG_MOBIL                  1
FLAG_WORK_PHONE             2
FLAG_PHONE                  2
FLAG_EMAIL                  2
OCCUPATION_TYPE            18
CNT_FAM_MEMBERS            13
dtype: int64

In [7]:
if "OCCUPATION_TYPE" in application_record.columns:
    application_record = application_record.drop(columns=["OCCUPATION_TYPE"])

In [8]:
application_record["CODE_GENDER"] = application_record["CODE_GENDER"].map({"F":0, "M":1})
application_record["FLAG_OWN_CAR"] = application_record["FLAG_OWN_CAR"].map({"N":0, "Y":1})
application_record["FLAG_OWN_REALTY"] = application_record["FLAG_OWN_REALTY"].map({"N":0, "Y":1})

In [9]:
application_record["AGE"] = (-application_record["DAYS_BIRTH"] / 365).astype(int)
application_record["EXPERIENCE"] = application_record["DAYS_EMPLOYED"].apply(lambda x: 0 if x > 0 else int(-x/365))
application_record["CNT_FAM_MEMBERS"] = application_record["CNT_FAM_MEMBERS"].round().astype(int)

In [10]:
cat_cols = ["NAME_INCOME_TYPE","NAME_EDUCATION_TYPE","NAME_FAMILY_STATUS","NAME_HOUSING_TYPE"]
application_record = pd.get_dummies(application_record, columns=cat_cols, drop_first=True)

In Credit_record STATUS:
- 0,1,2,3,4,5 - debt levels
- C - closed loan
- X - no loan
- We consider "bad" if status is 2,3,4,5

In [11]:
credit_record["bad"] = credit_record["STATUS"].apply(lambda x: 1 if x in ["2","3","4","5"] else 0)
client_status = credit_record.groupby("ID")["bad"].max().reset_index()

In [12]:
data = application_record.merge(client_status, on="ID", how="inner")

In [13]:
print("target value counts:")
print(data["bad"].value_counts())

target value counts:
bad
0    35841
1      616
Name: count, dtype: int64


# FEATURES 

In [14]:
X = data.drop(columns=["ID", "DAYS_BIRTH", "DAYS_EMPLOYED", "bad"])
y = data["bad"]

In [15]:
X_train_hold, X_test_hold, y_train_hold, y_test_hold = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 12,
    random_state = 42,
    n_jobs=-1
)

clf.fit(X_train_hold, y_train_hold)

train_acc = accuracy_score(y_train_hold, clf.predict(X_train_hold))
test_acc  = accuracy_score(y_test_hold, clf.predict(X_test_hold))

print("Train accuracy:", train_acc)
print("Test accuracy:",  test_acc)

Train accuracy: 0.9840905194582548
Test accuracy: 0.9827207899067472


# MODEL: RANDOM FOREST CLASSIFIER

In [16]:
model = RandomForestClassifier(
    n_estimators = 250,
    max_depth = 12,
    random_state = 42,
    n_jobs = -1
)
%time model.fit(X, y)

CPU times: user 3.08 s, sys: 103 ms, total: 3.18 s
Wall time: 606 ms


RandomForestClassifier(max_depth=12, n_estimators=250, n_jobs=-1,
                       random_state=42)

# 5-fold CV

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print("Cross-validation accuracies:", scores)
print("Mean CV accuracy:", scores.mean())

Cross-validation accuracies: [0.98244652 0.98258365 0.98312989 0.98285558 0.98312989]
Mean CV accuracy: 0.9828291035476603


In [18]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

In [19]:
least = importances.tail(5).index
X_reduced = X.drop(columns=least)

scores_reduced = cross_val_score(model, X_reduced, y, cv=skf, scoring='accuracy')
print(scores_reduced.mean())

0.9828291073094647


In [20]:
from sklearn.metrics import roc_auc_score

prob = clf.predict_proba(X_test_hold)[:,1]
auc = roc_auc_score(y_test_hold, prob)
print("AUC:", auc)

AUC: 0.7174652155225696


In [21]:
def specificity_score(y_true, y_pred):
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    return tn / (tn + fp) if (tn+fp)>0 else 0

In [22]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_list = []
rec_list = []
prc_list = []
spe_list = []
f1_list  = []

for train_index, valid_index in skf.split(X, y):
    X_tr, X_val = X.iloc[train_index], X.iloc[valid_index]
    y_tr, y_val = y.iloc[train_index], y.iloc[valid_index]

    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)

    acc_list.append(accuracy_score(y_val, preds))
    rec_list.append(recall_score(y_val, preds))
    prc_list.append(precision_score(y_val, preds))
    spe_list.append(specificity_score(y_val, preds))
    f1_list.append(f1_score(y_val, preds))

print("Accuracy: {:.2f}%".format(np.mean(acc_list)*100))
print("Recall: {:.2f}%".format(np.mean(rec_list)*100))
print("Precision: {:.2f}%".format(np.mean(prc_list)*100))
print("Specificity: {:.2f}%".format(np.mean(spe_list)*100))
print("F1-score: {:.2f}%".format(np.mean(f1_list )*100))


Accuracy: 98.28%
Recall: 1.14%
Precision: 31.11%
Specificity: 99.95%
F1-score: 2.18%


In [25]:
# Let's define our first Random Forest Classifier
classifier = RandomForestClassifier(
    n_estimators = 20, 
    class_weight = 'balanced', # classifier specific
    criterion = 'gini',  # classifier specific
    max_depth = 3, 
    min_samples_split = 5, 
    min_samples_leaf = 2, 
    min_weight_fraction_leaf = 0.0, 
    max_features = None,
    max_leaf_nodes = None, 
    min_impurity_decrease = 0.0,
    bootstrap = True, 
    oob_score = True, 
    max_samples = 10000,
    random_state = 42,
)

In [26]:
acc_list = []
rec_list = []
prc_list = []
spe_list = []
f1_list  = []

for train_index, valid_index in skf.split(X, y):
    X_tr, X_val = X.iloc[train_index], X.iloc[valid_index]
    y_tr, y_val = y.iloc[train_index], y.iloc[valid_index]

    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)

    acc_list.append(accuracy_score(y_val, preds))
    rec_list.append(recall_score(y_val, preds))
    prc_list.append(precision_score(y_val, preds))
    spe_list.append(specificity_score(y_val, preds))
    f1_list.append(f1_score(y_val, preds))

print("Accuracy   (mean): {:.2f}%".format(np.mean(acc_list)*100))
print("Recall     (mean): {:.2f}%".format(np.mean(rec_list)*100))
print("Precision  (mean): {:.2f}%".format(np.mean(prc_list)*100))
print("Specificity(mean): {:.2f}%".format(np.mean(spe_list)*100))
print("F1-score   (mean): {:.2f}%".format(np.mean(f1_list )*100))

Accuracy   (mean): 98.28%
Recall     (mean): 1.14%
Precision  (mean): 31.11%
Specificity(mean): 99.95%
F1-score   (mean): 2.18%
